# Notebook 07 — Überlappende Fenster + SVM

**Hypothese:** Mehr Trainingsdaten durch überlappende Fenster verbessern die SVM-Genauigkeit.

**Wichtig — kein Data Leakage:**  
Mit überlappenden Fenstern teilen benachbarte Fenster ~90 % ihrer Daten. Deshalb wird
**Leave-One-Session-Out (LOSO)** verwendet: immer alle Fenster einer kompletten Aufnahme
rauslassen. Das ist strenger als NB04/05 (LOO per Fenster), aber ehrlich.

```
Kein Overlap (NB04):  |── 25 s ──|── 25 s ──|── 25 s ──|
5 s Step (80 %):      |── 25 s ──|
                           |── 25 s ──|
                                |── 25 s ──|
```

**Verglichene Schrittweiten:** 25 s · 12.5 s · 5 s · 2.5 s

---
## 1. Setup

In [ ]:
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

CLASSES_RAW    = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_COARSE = ['Still', 'Essen']
CLASSES_FINE   = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
CLASSES_FULL   = ['Still', 'Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE      = {'Apfel': 'Essen', 'Kaugummi': 'Essen',
                  'Skyr':  'Essen', 'Still':    'Still', 'Essen': 'Essen'}
COLORS = {'Apfel': '#e15759', 'Kaugummi': '#4e79a7', 'Skyr': '#f28e2b',
          'Still': '#59a14f', 'Essen': '#b07aa1'}

FS            = 50.0
TRIM_SECS     = 2
WINDOW_SECS   = 25.0
MIN_TAIL_SECS = 20.0
K_ME          = 5

SVM_PARAMS = dict(kernel='rbf', C=10, probability=True,
                  class_weight='balanced', random_state=42)

STEP_SIZES = [25.0, 12.5, 5.0, 2.5]

# NB06-Referenz (SVM, kein Overlap, LOO per Fenster)
NB06_SVM_S2 = 0.930

---
## 2. Daten laden & vorverarbeiten

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) &
            (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x']     = df['accelerationX']
    df['lin_y']     = df['accelerationY']
    df['lin_z']     = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

def movement_mask(df):
    thr = max(0.02, K_ME * df['magnitude'].median())
    return df['magnitude'].rolling(50, center=True, min_periods=1).max() <= thr

def extract_features(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int((df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean']  = df[col].mean()
        feats[f'{col}_std']   = df[col].std()
        feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(256, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0)
    cf, cp = freqs[chew], psd[chew]
    feats['total_power']        = float(psd.sum())
    feats['chew_band_power']    = float(cp.sum())
    feats['rhythmicity']        = feats['chew_band_power'] / feats['total_power'] if feats['total_power'] > 0 else 0.0
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

sessions_per_class = {cls: [] for cls in CLASSES_RAW}
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            with zipfile.ZipFile(zf) as z:
                csv_name = next(f for f in z.namelist() if f.endswith('.csv') and f not in _SKIP)
                with z.open(csv_name) as f:
                    sessions_per_class[cls].append(preprocess(pd.read_csv(f)))
            break

print('Sitzungen pro Klasse:')
for cls in CLASSES_RAW:
    n    = len(sessions_per_class[cls])
    durs = [df['seconds_elapsed'].iloc[-1] - df['seconds_elapsed'].iloc[0]
            for df in sessions_per_class[cls]]
    print(f'  {cls:12s}: {n:2d} Sitzungen  ({sum(durs):.0f} s gesamt)')

---
## 3. Sliding-Window-Funktion & Datensatz-Builder

In [ ]:
def sliding_windows(df, window_secs=WINDOW_SECS, step_secs=5.0, min_secs=MIN_TAIL_SECS):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    out = []
    while t_start + min_secs <= t_end:
        t_stop = t_start + window_secs
        w = df[(t >= t_start) & (t < t_stop)].reset_index(drop=True)
        if len(w) > 1 and (w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]) >= min_secs:
            out.append(w)
        t_start += step_secs
    return out

def build_dataset(step_secs):
    """Feature-Matrix + Labels + Session-IDs fuer gegebene Schrittweite."""
    rows, y_list, session_ids = [], [], []
    session_counter = 0
    for cls in CLASSES_RAW:
        for df in sessions_per_class[cls]:
            for w in sliding_windows(df, step_secs=step_secs):
                clean = w[movement_mask(w)].reset_index(drop=True)
                if len(clean) > 50:
                    rows.append(extract_features(clean))
                    y_list.append(cls)
                    session_ids.append(session_counter)
            session_counter += 1
    X    = pd.DataFrame(rows)
    y    = np.array(y_list)
    grps = np.array(session_ids)
    return X, y, grps

print(f'{"Step":>6s}  {"Gesamt":>7s}  {"Still":>6s}  {"Apfel":>6s}  {"Kaugummi":>9s}  {"Skyr":>6s}  {"Essen":>6s}')
print('─' * 60)
for step in STEP_SIZES:
    X_tmp, y_tmp, _ = build_dataset(step)
    yc = np.array([TO_COARSE[c] for c in y_tmp])
    print(f'{step:>5.1f}s  {len(y_tmp):>7d}  {(yc=="Still").sum():>6d}  '
          f'{(y_tmp=="Apfel").sum():>6d}  {(y_tmp=="Kaugummi").sum():>9d}  '
          f'{(y_tmp=="Skyr").sum():>6d}  {(y_tmp=="Essen").sum():>6d}')

---
## 4. LOSO-CV — Stufe 2 Vergleich

**Leave-One-Session-Out:** alle Fenster einer Aufnahme gleichzeitig als Test-Set.
Verhindert Leakage durch überlappende Fenster aus derselben Aufnahme.

In [ ]:
def loso_svm_s2(X_np, y, grps):
    yt_all, yp_all = [], []
    for tr, te in LeaveOneGroupOut().split(X_np, y, groups=grps):
        if len(np.unique(y[tr])) < 2:
            continue
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_np[tr])
        Xte = sc.transform(X_np[te])
        clf = SVC(**SVM_PARAMS)
        clf.fit(Xtr, y[tr])
        yp_all.extend(clf.predict(Xte).tolist())
        yt_all.extend(y[te].tolist())
    return np.array(yt_all), np.array(yp_all)

loso_results = {}
for step in STEP_SIZES:
    print(f'Stufe 2 Step={step:.1f}s…', end=' ', flush=True)
    X, y, grps = build_dataset(step)
    yc         = np.array([TO_COARSE[c] for c in y])
    eat_mask   = yc == 'Essen'
    yt, yp     = loso_svm_s2(X[eat_mask].values, y[eat_mask], grps[eat_mask])
    acc        = accuracy_score(yt, yp)
    loso_results[step] = {
        'acc': acc, 'yt': yt, 'yp': yp,
        'n_eat': eat_mask.sum(),
        'X': X, 'y': y, 'grps': grps
    }
    print(f'{acc:.1%}  ({eat_mask.sum()} Essen-Fenster)')

best_step = max(loso_results, key=lambda s: loso_results[s]['acc'])

---
## 5. End-to-End LOSO — Stufe 1 + Stufe 2

In [ ]:
def loso_e2e(X_np, y_str, grps):
    y_coarse = np.array([TO_COARSE[c] for c in y_str])
    eat_mask = y_coarse == 'Essen'
    yt_all, yp_all = [], []
    for tr, te in LeaveOneGroupOut().split(X_np, y_str, groups=grps):
        if len(np.unique(y_coarse[tr])) < 2:
            continue
        # Stufe 1
        sc1  = StandardScaler()
        Xtr1 = sc1.fit_transform(X_np[tr])
        Xte1 = sc1.transform(X_np[te])
        m1   = SVC(**SVM_PARAMS)
        m1.fit(Xtr1, y_coarse[tr])
        preds_c = m1.predict(Xte1)

        # Stufe 2 vorbereiten
        eat_tr_idx = np.intersect1d(tr, np.where(eat_mask)[0])
        if len(eat_tr_idx) > 0 and len(np.unique(y_str[eat_tr_idx])) >= 2:
            sc2  = StandardScaler()
            Xtr2 = sc2.fit_transform(X_np[eat_tr_idx])
            m2   = SVC(**SVM_PARAMS)
            m2.fit(Xtr2, y_str[eat_tr_idx])
        else:
            m2 = sc2 = None

        for local_i, (global_i, pred_c) in enumerate(zip(te, preds_c)):
            if pred_c == 'Essen' and m2 is not None:
                pred = m2.predict(sc2.transform(X_np[[global_i]]))[0]
            else:
                pred = pred_c
            yp_all.append(pred)
            yt_all.append(y_str[global_i])
    return np.array(yt_all), np.array(yp_all)

e2e_results = {}
for step in STEP_SIZES:
    r = loso_results[step]
    print(f'E2E Step={step:.1f}s…', end=' ', flush=True)
    yt, yp = loso_e2e(r['X'].values, r['y'], r['grps'])
    acc    = accuracy_score(yt, yp)
    e2e_results[step] = {'acc': acc, 'yt': yt, 'yp': yp}
    print(f'{acc:.1%}')

---
## 6. Ergebnisübersicht

In [ ]:
base_s2 = loso_results[25.0]['acc']
print(f'{"Step":>6s}  {"Essen-Fenster":>13s}  {"Stufe 2":>9s}  {"End-to-End":>11s}  {"Δ S2":>6s}')
print('─' * 55)
for step in STEP_SIZES:
    s2  = loso_results[step]['acc']
    e2e = e2e_results[step]['acc']
    n   = loso_results[step]['n_eat']
    mark = ' ←' if step == best_step else ''
    print(f'{step:>5.1f}s  {n:>13d}  {s2:>9.1%}  {e2e:>11.1%}  {s2 - base_s2:>+5.1%}{mark}')
print('─' * 55)
print(f'NB06 (LOO/Fenster, kein Overlap): S2={NB06_SVM_S2:.1%}  [andere CV → nicht direkt vergleichbar]')

---
## 7. Visualisierung

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Links: Accuracy vs. Step-Größe ───────────────────────────────────────────
ax = axes[0]
s2_acc = [loso_results[s]['acc'] for s in STEP_SIZES]
e2_acc = [e2e_results[s]['acc']  for s in STEP_SIZES]
ax.plot(STEP_SIZES, s2_acc, 'o-', color='#4e79a7', linewidth=2, label='Stufe 2')
ax.plot(STEP_SIZES, e2_acc, 's--', color='#e15759', linewidth=2, label='End-to-End')
ax.set_xlabel('Step-Größe (s)  →  mehr Overlap')
ax.set_ylabel('LOSO-Accuracy')
ax.set_title('Accuracy vs. Overlap', fontweight='bold')
ax.invert_xaxis()
ax.legend()
for s, a in zip(STEP_SIZES, s2_acc):
    ax.annotate(f'{a:.1%}', (s, a), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

# ── Mitte: Fensteranzahl vs. Step ────────────────────────────────────────────
ax2 = axes[1]
n_wins = [loso_results[s]['n_eat'] for s in STEP_SIZES]
bars   = ax2.bar([f'{s}s' for s in STEP_SIZES], n_wins,
                 color=['#4e79a7' if s == best_step else '#aec7e8' for s in STEP_SIZES],
                 edgecolor='white')
for bar, n in zip(bars, n_wins):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(n), ha='center', va='bottom', fontsize=10)
ax2.set_xlabel('Step-Größe')
ax2.set_ylabel('Essen-Fenster (Training)')
ax2.set_title('Trainingsdatenmenge', fontweight='bold')

# ── Rechts: Confusion Matrix bestes Modell ───────────────────────────────────
ax3 = axes[2]
r   = loso_results[best_step]
cm  = confusion_matrix(r['yt'], r['yp'], labels=CLASSES_FINE)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES_FINE, yticklabels=CLASSES_FINE, ax=ax3,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 12, 'weight': 'bold'}, vmin=0)
ax3.set_title(f'Stufe 2 — Step={best_step}s  ({r["acc"]:.1%})', fontweight='bold')
ax3.set_xlabel('Vorhergesagt'); ax3.set_ylabel('Tatsächlich')

plt.suptitle('NB07 — Überlappende Fenster + SVM  (LOSO-CV)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb07_sliding_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r = loso_results[best_step]
print(f'Detail-Report — SVM Stufe 2, Step={best_step}s (LOSO-CV):')
print(classification_report(r['yt'], r['yp'], labels=CLASSES_FINE, zero_division=0))

re = e2e_results[best_step]
print(f'Detail-Report — End-to-End, Step={best_step}s (LOSO-CV):')
print(classification_report(re['yt'], re['yp'], labels=CLASSES_FULL, zero_division=0))